In [36]:
import pandas as pd
import json
from kafka import KafkaProducer
from hw_models import ride_serializer, ride_from_row
from time import time


In [37]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"
columns = ["lpep_pickup_datetime", "lpep_dropoff_datetime", "PULocationID", "DOLocationID", "passenger_count", "trip_distance", "tip_amount", "total_amount"]
df = pd.read_parquet(url, columns=columns)

In [38]:
df.shape[0]

49416

In [39]:
datetime_cols = df.select_dtypes(include=["datetime64[ns]", "datetimetz"]).columns

df[datetime_cols] = df[datetime_cols].astype(str)

In [40]:
row = df.iloc[0]

In [41]:
row

lpep_pickup_datetime     2025-10-01 00:21:47
lpep_dropoff_datetime    2025-10-01 00:24:37
PULocationID                             247
DOLocationID                              69
passenger_count                          1.0
trip_distance                            0.7
tip_amount                               1.7
total_amount                            10.0
Name: 0, dtype: object

In [42]:
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=ride_serializer
)
topic = "green-trips"

In [43]:
t0 = time()
for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic, value=ride)
producer.flush()

t1 = time()
print(f'took {(t1 - t0):.2f} seconds')

took 6.55 seconds
